# Work with Measurement Service

This notebook shows what can be done with the Measurement Service.

In [1]:
%cd ../
%load_ext autoreload
%autoreload 2

/Users/matthaei/Documents/code/python/bachelor-project


In [2]:
MIGRATE_DATABASE = False

In [3]:
from src.measurements.measurement_service import MeasurementService
from src.weather_stations.weather_station_service import WeatherStationService
from src.database.database_service import DatabaseService
from omegaconf import DictConfig, OmegaConf
from hydra import compose, initialize_config_dir
import os
from datetime import datetime

/Users/matthaei/Documents/code/python/bachelor-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Initialize Hydra configuration
config_dir = os.path.abspath("./conf")

# Initialize Hydra with the config directory
with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(config_name="config")

In [5]:
database_service = DatabaseService(cfg)

if MIGRATE_DATABASE:
    database_service.create_tables()

In [6]:
weather_station_service = WeatherStationService(cfg, database_service)
weather_stations_df = weather_station_service.load_from_database()

2025-09-10 10:37:13.449 | INFO     | src.weather_stations.weather_station_data_provider:load_from_database:228 - Loaded 50 weather stations from database


## Fill data from DWD

In [7]:
measurement_service = MeasurementService(cfg, database_service, weather_stations_df)

In [10]:
df = measurement_service.fill_database_with_measurements(only_now=False)

2025-09-10 10:46:52.781 | INFO     | src.measurements.measurement_data_provider:_download_file:379 - Downloaded file from https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/wind/historical/10minutenwerte_wind_00096_20200101_20241231_hist.zip as dataframe with 263088 rows
2025-09-10 10:46:53.061 | INFO     | src.measurements.measurement_data_provider:_download_file:379 - Downloaded file from https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/wind/recent/10minutenwerte_wind_00096_akt.zip as dataframe with 79200 rows
2025-09-10 10:46:53.174 | INFO     | src.measurements.measurement_data_provider:_download_file:379 - Downloaded file from https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/wind/now/10minutenwerte_wind_00096_now.zip as dataframe with 48 rows
2025-09-10 10:46:53.698 | INFO     | src.measurements.measurement_data_provider:_download_file:379 - Downloaded file from https

## Load for datetime

In [ ]:
test_time = datetime(2025, 9, 4, 10, 0)
df = measurement_service.load_measurements_from_database_for_datetime(test_time)

2025-09-04 16:21:05.205 | INFO     | src.measurements.measurement_data_provider:load_measurements_from_database_for_datetime:200 - Loaded 17 measurements from database for datetime 2025-09-04 10:00:00


In [ ]:
df['station_id'].unique()

array([  96,  164,  303,  427,  880, 1001, 1869, 2794, 2856, 3015, 3158,
       3376, 3987, 5546, 5825, 6253])